## Structured Data Cleaning

**Architecture Overview:**
This notebook executes an industrial-grade data cleaning and synchronization task. We read Parquet/Delta files directly from the MinIO (S3) data lake using Apache Spark, perform deep structural cleaning in distributed memory (primary key hardening, anomaly handling, dynamic field inference), and finally synchronize the data to the ClickHouse modern analytical data warehouse via high-concurrency direct writes (MapPartitions).

**Pipeline Steps:**
1. **Environment Initialization**: Mount the Spark Session and connect to MinIO and ClickHouse.
2. **Dynamic Metadata Scanning**: Automatically identify valid data directories in the S3 Landing Zone.
3. **Distributed Adaptive Cleaning**: Apply different deduplication rules for different business tables (e.g., full-field deduplication for vehicle tables, composite primary key deduplication for climate tables).
4. **High-Concurrency Ingestion**: Pre-build DDL on the Driver, and perform distributed direct writes to ClickHouse from the Executors.
5. **Post-Cleaning Validation**: Check physical metrics, data distribution, and real sample exploration.

**Importing Useful Libraries**

In [1]:
import os
import re
import ast
import boto3
import clickhouse_connect
import pyspark.sql.functions as F
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType, DateType, TimestampType, ArrayType
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv()
endpoint = os.getenv("MINIO_ENDPOINT")
access_key = os.getenv("MINIO_ACCESS_KEY")
secret_key = os.getenv("MINIO_SECRET_KEY")

# 2. Initialize Spark Session with S3 & ClickHouse bindings
DELTA_VERSION = "4.1.0" 
CLICKHOUSE_CONNECTOR_VERSION = "0.8.0" 

spark = SparkSession.builder \
    .appName("Production-Data-Warehouse-Pipeline") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", 
            f"org.apache.hadoop:hadoop-aws:3.3.4,"
            f"com.amazonaws:aws-java-sdk-bundle:1.12.262,"
            f"io.delta:delta-spark_2.13:{DELTA_VERSION},"
            f"com.clickhouse.spark:clickhouse-spark-runtime-3.4_2.13:{CLICKHOUSE_CONNECTOR_VERSION},"
            f"com.clickhouse:clickhouse-jdbc:0.6.5") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.hadoop.fs.s3a.endpoint", endpoint) \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.access.key", access_key) \
    .config("spark.hadoop.fs.s3a.secret.key", secret_key) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .getOrCreate()

# 3. Normalize Hadoop config values to prevent JVM parse errors
hadoop_conf = spark.sparkContext._jsc.hadoopConfiguration()
for item in hadoop_conf.iterator():
    key, value = item.getKey(), item.getValue()
    if isinstance(value, str) and (value.endswith("s") or value.endswith("h")):
        hadoop_conf.set(key, "".join(char for char in value if char.isdigit()))

print("Spark Cluster environment initialized successfully.")

Spark Cluster environment initialized successfully.


In [2]:
# Access Hadoop underlying FileSystem API for efficient path scanning
sc = spark.sparkContext
base_path_str = "s3a://landing-zone/persistent-landing/structured/"
path_obj = sc._jvm.org.apache.hadoop.fs.Path(base_path_str)
fs = path_obj.getFileSystem(sc._jsc.hadoopConfiguration())

# Filter out hidden folders and raw backup directories, capturing valid Delta/Parquet paths
valid_delta_paths = []
for status in fs.listStatus(path_obj):
    if status.isDirectory():
        full_path = status.getPath().toString()
        folder_name = full_path.split("/")[-1] or full_path.split("/")[-2]
        
        # Filter out non-business data directories
        if folder_name in ["raw", "file_catalog"] or folder_name.startswith("."):
            continue
            
        valid_delta_paths.append(full_path)

print(f"Discovered {len(valid_delta_paths)} analytical dataset targets in S3 Landing Zone.")

Discovered 4 analytical dataset targets in S3 Landing Zone.


### Core Business Logic
The following defines two core industrial-grade functions:
1. `extract_clean_dataframe_and_ddl`: Handles structural anomalies, dynamically generates DDL, applies multi-dimensional composite primary key protection, and processes extreme null value distributions.
2. `load_dataframe_to_clickhouse_parallel`: Bypasses the slow Driver node collection and directly instructs the backend Spark Workers to push memory data into ClickHouse in parallel.

In [3]:
def extract_clean_dataframe_and_ddl(spark_session: SparkSession, s3_path: str, database_name: str = "bi_analytics") -> tuple[DataFrame, str]:
    """Reads, cleans structured data, and dynamically generates ClickHouse DDL with robust error handling."""
    import re
    from pyspark.sql.types import StringType, IntegerType, LongType, FloatType, DoubleType, BooleanType, DateType, TimestampType, ShortType, ByteType, ArrayType
    import pyspark.sql.functions as F

    # 1. Automatically extract and normalize the target table name
    folder_name = s3_path.split("/")[-1] or s3_path.split("/")[-2]
    clean_table_name = re.sub(r'_\d+$', '', folder_name.replace("_delta", "")).replace("-", "_").lower()
    
    # 2. Adaptively read Delta or Parquet format based on _delta_log existence
    sc = spark_session.sparkContext
    delta_log_path = sc._jvm.org.apache.hadoop.fs.Path(s3_path.rstrip("/") + "/_delta_log")
    fs = delta_log_path.getFileSystem(sc._jsc.hadoopConfiguration())
    df = spark_session.read.format("delta").load(s3_path) if fs.exists(delta_log_path) else spark_session.read.parquet(s3_path)
        
    # 3. Clean column names (strip BOM headers, spaces, brackets) and standardize primary ID
    for col_name in df.columns:
        cleaned_name = col_name.replace("ï»¿", "").replace(" ", "_").replace("(", "").replace(")", "").replace("/", "_")
        if cleaned_name.lower() in ["tweet_id", "id"]:
            cleaned_name = "id"
        if cleaned_name != col_name:
            df = df.withColumnRenamed(col_name, cleaned_name)

    # to freeze their current precision state and prevent further degradation.
    if "id" in df.columns and "tweet" in clean_table_name:
        df = df.withColumn("id", F.col("id").cast("string"))

    if "temperature_change" in clean_table_name and "Unit" in df.columns:
        df = df.withColumn("Unit", F.when(F.col("Unit").contains("â"), F.lit("°C")).otherwise(F.col("Unit")))

    # 4. Sanitize text fields and strip hidden control characters (\n, \r, \t) safely
    string_cols = [f.name for f in df.schema.fields if isinstance(f.dataType, StringType)]
    for c in string_cols:
        df = df.withColumn(c, F.lower(F.trim(F.regexp_replace(F.col(c), r'[\n\r\t]', ' '))))

    # 5. Floating-point hardening: Intercept and mask NaN or Infinity exceptions
    for c in [f.name for f in df.schema.fields if isinstance(f.dataType, (FloatType, DoubleType))]:
        df = df.withColumn(c, F.when(F.isnan(F.col(c)) | F.col(c).cast("string").contains("Infinity"), F.lit(None)).otherwise(F.col(c)))

    # 6. Semi-structured array cleaning (Transform pseudo-array strings "['a', 'b']" into physical arrays)
    for array_col in ["hashtags", "emojis"]:
        if array_col in df.columns:
            if "string" in dict(df.dtypes)[array_col]:
                df = df.withColumn(array_col, F.split(F.regexp_replace(F.col(array_col), r'[\[\]\'"\s]', ''), ","))
            df = df.withColumn(array_col, F.expr(f"filter(transform({array_col}, x -> lower(trim(x))), x -> x IS NOT NULL AND x != '')"))
            
    # 7. Smart primary key routing for MergeTree ORDER BY clause
    sorting_keys = []
    if "global_warming" in clean_table_name:
        sorting_keys = ["Country", "Year"]
    elif "temperature_change" in clean_table_name:
        sorting_keys = ["Area", "Months", "Year"]
    elif "emission" in clean_table_name:
        sorting_keys = ["Make", "Model"]
    elif "tweet" in clean_table_name:
        sorting_keys = ["id"]

    final_sorting_keys = [k for k in sorting_keys if k in df.columns] or [k for k in ["id"] if k in df.columns]
    if not final_sorting_keys:
        final_sorting_keys = [df.columns[0]]

    for pk_col in final_sorting_keys:
        col_type = dict(df.dtypes)[pk_col]
        if "string" in col_type:
            df = df.withColumn(pk_col, F.coalesce(F.col(pk_col), F.lit("unknown")))
        elif "int" in col_type or "long" in col_type:
            df = df.withColumn(pk_col, F.coalesce(F.col(pk_col), F.lit(0)))
        elif "double" in col_type or "float" in col_type:
            df = df.withColumn(pk_col, F.coalesce(F.col(pk_col), F.lit(0.0)))
    
    if any(k in clean_table_name for k in ["emission", "global_warming", "temperature_change", "tweet"]):
        df = df.dropDuplicates()
    else:
        if final_sorting_keys:
            df = df.dropDuplicates(subset=final_sorting_keys)
        else:
            df = df.dropDuplicates()

    # 8. Dynamic ClickHouse DDL generation mapping (Comprehensive type topology translation)
    type_map = {
        StringType: "String", IntegerType: "Int32", LongType: "Int64",
        FloatType: "Float32", DoubleType: "Float64", BooleanType: "UInt8",
        DateType: "Date", TimestampType: "DateTime", ShortType: "Int16", ByteType: "Int8"
    }
    ch_columns = []
    
    for field in df.limit(0).schema.fields:
        is_arr = isinstance(field.dataType, ArrayType)
        if is_arr:
            element_type_class = type(field.dataType.elementType)
            ch_type = f"Array({type_map.get(element_type_class, 'String')})"
        else:
            ch_type = type_map.get(type(field.dataType), "String")
            
        # Primary keys and Array types must never be wrapped in Nullable() in ClickHouse
        if field.name in final_sorting_keys or is_arr:
            ch_columns.append(f"    `{field.name}` {ch_type}")
        else:
            ch_columns.append(f"    `{field.name}` Nullable({ch_type})")
        
    order_by_clause = ", ".join([f"`{k}`" for k in final_sorting_keys]) if final_sorting_keys else f"`{df.columns[0]}`"
    native_ddl = f"CREATE TABLE IF NOT EXISTS {database_name}.{clean_table_name} (\n{',\n'.join(ch_columns)}\n) ENGINE = MergeTree()\nORDER BY ({order_by_clause})"

    return df, native_ddl


def load_dataframe_to_clickhouse_parallel(df: DataFrame, ddl_sql: str, target_table_name: str, database_name: str = "bi_analytics"):
    """Uses MapPartitions to concurrently write Spark cluster data directly to ClickHouse."""
    
    # Driver executes table creation DDL
    driver_client = clickhouse_connect.get_client(host='clickhouse', port=8123, username='analytics', password='analytics_secret', database=database_name)
    driver_client.command(f"DROP TABLE IF EXISTS {database_name}.{target_table_name}") # Force schema refresh
    driver_client.command(ddl_sql)
    driver_client.close()

    column_names = df.columns

    # Executor-side direct write closure
    def send_partition(partition_iterator):
        worker_client = clickhouse_connect.get_client(host='clickhouse', port=8123, username='analytics', password='analytics_secret', database=database_name)
        payload = [tuple(row) for row in partition_iterator]
        rows_written = 0
        if payload:
            worker_client.insert(table=target_table_name, data=payload, column_names=column_names)
            rows_written = len(payload)
        worker_client.close()
        yield rows_written

    # Trigger execution and calculate total rows
    total_inserted = df.rdd.mapPartitions(send_partition).sum()
    print(f"Parallel sync completed for '{target_table_name}'. Total rows ingested: {total_inserted}")

In [4]:
print("Starting Automated Data Cleaning & Warehouse Ingestion Pipeline")

# Iterate through valid data directories and execute the full pipeline
for s3_delta_path in valid_delta_paths:
    folder_name = s3_delta_path.split("/")[-1] or s3_delta_path.split("/")[-2]
    tbl_name = re.sub(r'_\d+$', '', folder_name.replace("_delta", "")).replace("-", "_").lower()

    print(f"\nProcessing target: {tbl_name} ...")

    try:
        # 1. Deep clean and generate DDL
        cleaned_df, clickhouse_ddl = extract_clean_dataframe_and_ddl(spark, s3_delta_path)
        
        # 2. Distributed direct ingestion (includes automatic table clearing)
        load_dataframe_to_clickhouse_parallel(cleaned_df, clickhouse_ddl, target_table_name=tbl_name)
        
    except Exception as e:
        print(f"Synchronizer halted processing target table '{tbl_name}'. Error: {e}")
        continue
        
print("\nAll production datasets have been successfully synced to the Data Warehouse.")

Starting Automated Data Cleaning & Warehouse Ingestion Pipeline

Processing target: co2_emission_by_vehicles ...
Parallel sync completed for 'co2_emission_by_vehicles'. Total rows ingested: 5990

Processing target: global_warming_dataset ...
Parallel sync completed for 'global_warming_dataset'. Total rows ingested: 100000

Processing target: natural_disaster_tweets ...
Parallel sync completed for 'natural_disaster_tweets'. Total rows ingested: 127534

Processing target: temperature_change ...
Parallel sync completed for 'temperature_change'. Total rows ingested: 241893

All production datasets have been successfully synced to the Data Warehouse.


### Post-Cleaning Data Validation
This testing module connects to the final ClickHouse physical data warehouse layer to verify:
1. **Physical Storage Metrics**: Check whether the stored row count has severely shrunk/expanded, and monitor disk space utilization.
2. **Data Sampling**: Visually verify that composite primary keys and nested arrays meet ClickHouse's business requirements.

In [5]:
# 1. Establish ClickHouse connection
client = clickhouse_connect.get_client(
    host='clickhouse', 
    port=8123, 
    username='analytics', 
    password='analytics_secret', 
    database='bi_analytics'
)

# 2. Physical Storage Metrics
metrics_query = """
SELECT 
    table AS `table_name`,
    sum(rows) AS `total_physical_rows`,
    formatReadableSize(sum(bytes_on_disk)) AS `disk_usage_size`
FROM system.parts
WHERE database = 'bi_analytics' AND active = 1
GROUP BY table
"""
print("Data Warehouse Storage Metrics (ClickHouse)")
for row in client.query(metrics_query).result_rows:
    print(f"Table: {row[0]:<30} | Rows: {row[1]:<10} | Disk Storage: {row[2]}")

# 3. Real Data Sampling Preview (Using Pandas to prevent any truncation)
print("Target Table Data Sampling Preview (Top 3 Rows - Untruncated)")

import pandas as pd
# Force Pandas to display absolute complete text and columns without wrapping or dot-dot-dot
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 1000)

target_tables = ['natural_disaster_tweets', 'co2_emission_by_vehicles', 'temperature_change', 'global_warming_dataset']

for table in target_tables:
    try:
        # Fetch actual column headers
        cols_query = f"SELECT name FROM system.columns WHERE database = 'bi_analytics' AND table = '{table}'"
        headers = [row[0] for row in client.query(cols_query).result_rows]
        
        # Fetch top 3 rows natively
        data_query = f"SELECT * FROM bi_analytics.{table} LIMIT 3"
        data_res = client.query(data_query)
        
        print(f"\nTable: {table}")
        print("-" * 80)
        
        if not data_res.result_rows:
            print("  Empty: No data available in ClickHouse.")
        else:
            # Build an explicit DataFrame to let Pandas handle the clean tabular rendering
            pdf = pd.DataFrame(data_res.result_rows, columns=headers)
            
            # Formatter: Convert native arrays back to readable lists for visualization
            for col in pdf.columns:
                pdf[col] = pdf[col].apply(lambda x: list(x) if isinstance(x, (list, tuple)) else x)
                
            display(pdf)
            
    except Exception as e:
        # Print warning if a specific table schema setup crashed
        print(f"Could not preview table '{table}': {e}")

client.close()

Data Warehouse Storage Metrics (ClickHouse)
Table: natural_disaster_tweets        | Rows: 127534     | Disk Storage: 12.86 MiB
Table: co2_emission_by_vehicles       | Rows: 5990       | Disk Storage: 130.18 KiB
Table: temperature_change             | Rows: 241893     | Disk Storage: 2.23 MiB
Table: global_warming_dataset         | Rows: 100000     | Disk Storage: 18.12 MiB
Target Table Data Sampling Preview (Top 3 Rows - Untruncated)

Table: natural_disaster_tweets
--------------------------------------------------------------------------------


,id,tweet_text,disaster_type,hashtags,emojis
0,1.00113669658963149e18,"flash floods struck a maryland city on sunday, washing out streets and tossing cars like bath toys.",flood,[],[]
1,1.0011369503451095e18,state of emergency declared for maryland flooding: via @youtube,flood,[],[]
2,1.00113932307541606e18,rt @dwnews: flash floods have turned the main street in ellicott city in maryland into a raging river,flood,[],[]



Table: co2_emission_by_vehicles
--------------------------------------------------------------------------------


,Make,Model,Vehicle_Class,Engine_SizeL,Cylinders,Transmission,Fuel_Type,Fuel_Consumption_City_L_100_km,Fuel_Consumption_Hwy_L_100_km,Fuel_Consumption_Comb_L_100_km,Fuel_Consumption_Comb_mpg,CO2_Emissionsg_km
0,acura,ilx,compact,2.4,4,am8,z,9.9,7.0,8.6,33,199
1,acura,ilx,compact,2.0,4,as5,z,9.7,6.7,8.3,34,191
2,acura,ilx,compact,2.4,4,am8,z,9.3,6.6,8.1,35,189



Table: temperature_change
--------------------------------------------------------------------------------


,Domain_Code,Domain,Area_Code_M49,Area,Element_Code,Element,Months_Code,Months,Year_Code,Year,Unit,Value,Flag,Flag_Description
0,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1962,1962,â°c,0.040,e,estimated value
1,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1965,1965,â°c,-1.850,e,estimated value
2,et,temperature change on land,4,afghanistan,7271,temperature change,7004,april,1969,1969,â°c,-0.813,e,estimated value



Table: global_warming_dataset
--------------------------------------------------------------------------------


,Country,Year,Temperature_Anomaly,CO2_Emissions,Population,Forest_Area,GDP,Renewable_Energy_Usage,Methane_Emissions,Sea_Level_Rise,Arctic_Ice_Extent,Urbanization,Deforestation_Rate,Extreme_Weather_Events,Average_Rainfall,Solar_Energy_Potential,Waste_Management,Per_Capita_Emissions,Industrial_Activity,Air_Pollution_Index,Biodiversity_Index,Ocean_Acidification,Fossil_Fuel_Usage,Energy_Consumption_Per_Capita,Policy_Score,Average_Temperature
0,country_1,1901,1.043901,9.882580e+08,5.083236e+08,84.965735,6.917356e+11,32.832251,2.878821e+06,15.553877,10.401744,89.184323,2.200617,24,2047.296831,1939.850470,48.212037,1.413663,20.970578,88.623903,94.277594,8.191214,96.483974,3601.007397,16.003608,7.210238
1,country_1,1903,-0.036591,8.441318e+08,6.914867e+08,71.224647,5.115886e+12,40.859095,1.266390e+06,-2.300988,4.807944,80.122040,3.800718,6,3258.582936,608.673754,7.183343,0.613889,95.199743,118.241051,90.210194,7.642089,38.464649,2135.170839,2.196041,-7.285549
2,country_1,1904,-1.319920,9.619559e+08,7.383737e+08,39.551921,4.918382e+12,57.245417,3.647545e+06,0.858311,8.809201,62.026295,1.911535,17,2036.448479,1694.309523,25.127089,2.086132,57.563322,185.585293,51.555234,7.702761,48.880101,4328.604536,63.683835,24.823660
